# Lab 2 — Country Schools: Preprocessing, Anomaly Detection & Clustering

**Dataset:** `Country Schools.csv` (school electrification / solar-kit programme per county in Kenya).

**Objectives**
1. Perform basic data preprocessing (clean, transform, engineer features).
2. Detect anomalous counties using univariate and multivariate methods.
3. Group counties into meaningful clusters and interpret the segments.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
RANDOM_STATE = 42

DATA_PATH = "Country Schools.csv"
print("Pandas", pd.__version__, "| NumPy", np.__version__)

Pandas 3.0.6 | NumPy 2.3.4


## 1. Load and inspect the raw data

In [3]:
df_raw = pd.read_csv(DATA_PATH)

print("Raw shape:", df_raw.shape)
print("\nColumn names:", list(df_raw.columns))
print("\nFirst 5 rows:")
display(df_raw.head())
print("\nLast 6 rows (note the blank rows and TOTAL row):")
display(df_raw.tail(6))
print("\nData types:")
display(df_raw.dtypes.rename("dtype").to_frame())
print("\nMissing values per column:")
display(df_raw.isna().sum().to_frame("missing"))

Raw shape: (50, 8)

Column names: ['SNO', 'TOTALSCHOOLS', 'TotalSchoolsInstalled', 'SchoolsPending', 'PercentageInstalled', 'InstalledLDDs', 'InstalledTDDs', 'FID']

First 5 rows:


,SNO,TOTALSCHOOLS,TotalSchoolsInstalled,SchoolsPending,PercentageInstalled,InstalledLDDs,InstalledTDDs,FID
0,TURKANA,358.00,37.00,321.00,10.34%,"3,744.00",74.00,1
1,HOMA BAY,875.00,552.00,323.00,63.09%,"22,982.00","1,104.00",2
2,WEST POKOT,498.00,242.00,256.00,48.59%,"15,114.00",484.00,3
3,KITUI,"1,238.00","1,200.00",38.00,96.93%,"42,162.00","2,400.00",4
4,TANA RIVER,159.00,143.00,16.00,89.94%,"7,415.00",286.00,5



Last 6 rows (note the blank rows and TOTAL row):


,SNO,TOTALSCHOOLS,TotalSchoolsInstalled,SchoolsPending,PercentageInstalled,InstalledLDDs,InstalledTDDs,FID
44,KAJIADO,395.00,232.00,163.00,58.73%,"8,893.00",464.00,45
45,EMBU,381.00,248.00,133.00,65.09%,"8,438.00",496.00,46
46,SAMBURU,146.00,49.00,97.00,33.56%,"2,891.00",98.00,47
47,TOTAL,"21,729.00","14,629.00","7,100.00",67.32%,"723,618.00","29,258.00",48
48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49
49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50



Data types:


,dtype
SNO,str
TOTALSCHOOLS,float64
TotalSchoolsInstalled,float64
SchoolsPending,float64
PercentageInstalled,str
InstalledLDDs,float64
InstalledTDDs,float64
FID,int64



Missing values per column:


,missing
SNO,2
TOTALSCHOOLS,2
TotalSchoolsInstalled,2
SchoolsPending,2
PercentageInstalled,2
InstalledLDDs,2
InstalledTDDs,2
FID,0
